# Faster-Whisper ASR Training for Twi
## High-Performance OpenAI Whisper Fine-tuning with CTranslate2

This notebook implements training and fine-tuning of Faster-Whisper models for Twi language ASR.

In [ ]:
# Installation and imports
!pip install -q faster-whisper transformers datasets torch torchaudio evaluate jiwer tensorboard
!pip install -q accelerate optimum ctranslate2

import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Audio processing
import torchaudio
import librosa
import soundfile as sf

# Whisper and training
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    WhisperTokenizer,
    WhisperFeatureExtractor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict, Audio
import evaluate

# Faster-whisper for inference
from faster_whisper import WhisperModel

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Auto-detect and use correct data path
import sys
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules

if IN_COLAB:
    print("🔍 Detected Google Colab environment")
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = Path('/content/drive/MyDrive/voice-lab-data')
elif IN_KAGGLE:
    print("🔍 Detected Kaggle environment")
    DATA_PATH = Path('/kaggle/working/voice-lab-data')
else:
    print("🔍 Detected local environment")
    DATA_PATH = Path('/opt/kasaai/apps/voice-lab/data')

# Check if processed data exists
metadata_path = DATA_PATH / 'metadata/all_datasets_processed.csv'
if metadata_path.exists():
    processed_data = pd.read_csv(metadata_path)
    print(f"✅ Found processed data: {len(processed_data)} samples")
    print(f"   Datasets: {processed_data['dataset'].unique()[:5]}...")
else:
    print("⚠️ Processed data not found at", metadata_path)
    print("   Please run notebooks 001 & 002 first to process the data")

# Configuration
@dataclass
class WhisperConfig:
    """Configuration for Whisper training."""
    # Model settings
    model_name: str = "openai/whisper-small"  # Options: tiny, base, small, medium, large-v3
    language: str = "tw"  # Twi language code
    task: str = "transcribe"
    
    # Data paths - now using auto-detected path
    data_dir: Path = DATA_PATH
    output_dir: Path = DATA_PATH / "models/whisper_twi"
    cache_dir: Path = DATA_PATH / "cache"
    
    # Training parameters
    batch_size: int = 16
    gradient_accumulation_steps: int = 2
    learning_rate: float = 1e-5
    warmup_steps: int = 500
    max_steps: int = 5000
    fp16: bool = True
    gradient_checkpointing: bool = True
    
    # Audio settings
    sample_rate: int = 16000
    max_duration: float = 30.0
    
    # Evaluation
    eval_steps: int = 500
    save_steps: int = 500
    logging_steps: int = 25
    
    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

config = WhisperConfig()
print(f"📋 Configuration loaded for {config.model_name}")
print(f"📁 Data directory: {config.data_dir}")

In [ ]:
# Data loading and preparation
class TwiDataLoader:
    """Load and prepare Twi datasets for Whisper training."""
    
    def __init__(self, config: WhisperConfig):
        self.config = config
        self.processor = WhisperProcessor.from_pretrained(
            config.model_name,
            language="tw",
            task="transcribe"
        )
        
    def load_datasets(self) -> DatasetDict:
        """Load all available Twi datasets."""
        datasets = []
        
        # Load Common Voice if available
        cv_path = self.config.data_dir / "processed/common_voice_twi_final"
        if cv_path.exists():
            print("📻 Loading Common Voice Twi...")
            cv_data = self._load_common_voice(cv_path)
            if cv_data:
                datasets.append(cv_data)
        
        # Load metadata CSV if available
        metadata_path = self.config.data_dir / "metadata/all_datasets_processed.csv"
        if metadata_path.exists():
            print("📊 Loading processed datasets from metadata...")
            meta_data = self._load_from_metadata(metadata_path)
            if meta_data:
                datasets.append(meta_data)
        
        # Combine all datasets
        if datasets:
            combined = self._combine_datasets(datasets)
            return self._create_splits(combined)
        else:
            print("⚠️ No datasets found! Creating dummy data for testing...")
            return self._create_dummy_data()
    
    def _load_common_voice(self, path: Path) -> Optional[Dataset]:
        """Load Common Voice dataset."""
        data = []
        
        # Look for audio files and transcripts
        audio_files = list(path.glob("**/*.wav")) + list(path.glob("**/*.mp3"))
        
        for audio_file in audio_files[:1000]:  # Limit for testing
            # Try to find corresponding transcript
            txt_file = audio_file.with_suffix('.txt')
            if txt_file.exists():
                with open(txt_file, 'r', encoding='utf-8') as f:
                    transcript = f.read().strip()
                
                data.append({
                    'audio': str(audio_file),
                    'text': transcript,
                    'dataset': 'common_voice'
                })
        
        if data:
            return Dataset.from_pandas(pd.DataFrame(data))
        return None
    
    def _load_from_metadata(self, metadata_path: Path) -> Optional[Dataset]:
        """Load dataset from metadata CSV."""
        df = pd.read_csv(metadata_path)
        
        # Filter for valid audio files
        if 'audio' in df.columns and 'processed_text' in df.columns:
            df = df.dropna(subset=['audio', 'processed_text'])
            
            # Check if audio files exist
            df['audio_exists'] = df['audio'].apply(lambda x: Path(x).exists() if pd.notna(x) else False)
            df = df[df['audio_exists']]
            
            if len(df) > 0:
                return Dataset.from_pandas(
                    df[['audio', 'processed_text']].rename(columns={'processed_text': 'text'})
                )
        return None
    
    def _create_dummy_data(self) -> DatasetDict:
        """Create dummy data for testing."""
        dummy_data = [
            {"text": "Me ma wo akye", "audio": "dummy1.wav"},
            {"text": "Wo ho te sɛn", "audio": "dummy2.wav"},
            {"text": "Me ho yɛ", "audio": "dummy3.wav"},
        ]
        
        dataset = Dataset.from_pandas(pd.DataFrame(dummy_data))
        return DatasetDict({
            'train': dataset,
            'validation': dataset,
            'test': dataset
        })
    
    def _combine_datasets(self, datasets: List[Dataset]) -> Dataset:
        """Combine multiple datasets."""
        from datasets import concatenate_datasets
        return concatenate_datasets(datasets)
    
    def _create_splits(self, dataset: Dataset, train_ratio: float = 0.8, val_ratio: float = 0.1) -> DatasetDict:
        """Create train/val/test splits."""
        # Shuffle dataset
        dataset = dataset.shuffle(seed=42)
        
        # Calculate split sizes
        total_size = len(dataset)
        train_size = int(total_size * train_ratio)
        val_size = int(total_size * val_ratio)
        
        # Create splits
        train_dataset = dataset.select(range(train_size))
        val_dataset = dataset.select(range(train_size, train_size + val_size))
        test_dataset = dataset.select(range(train_size + val_size, total_size))
        
        return DatasetDict({
            'train': train_dataset,
            'validation': val_dataset,
            'test': test_dataset
        })

# Load data
data_loader = TwiDataLoader(config)
datasets = data_loader.load_datasets()

print(f"\n📊 Dataset Statistics:")
for split, dataset in datasets.items():
    print(f"  {split}: {len(dataset)} samples")

In [ ]:
# Data preprocessing for Whisper
class WhisperPreprocessor:
    """Preprocess data for Whisper training."""
    
    def __init__(self, processor, config: WhisperConfig):
        self.processor = processor
        self.config = config
        self.feature_extractor = processor.feature_extractor
        self.tokenizer = processor.tokenizer
        
    def preprocess_function(self, examples):
        """Preprocess batch of examples."""
        # Load and resample audio
        audio_arrays = []
        
        for audio_path in examples["audio"]:
            if Path(audio_path).exists():
                audio, sr = librosa.load(audio_path, sr=self.config.sample_rate)
                # Limit duration
                max_samples = int(self.config.max_duration * self.config.sample_rate)
                if len(audio) > max_samples:
                    audio = audio[:max_samples]
                audio_arrays.append(audio)
            else:
                # Create dummy audio for missing files
                audio_arrays.append(np.zeros(self.config.sample_rate, dtype=np.float32))
        
        # Extract features
        inputs = self.feature_extractor(
            audio_arrays,
            sampling_rate=self.config.sample_rate,
            return_tensors="pt"
        )
        
        # Tokenize text
        labels = self.tokenizer(
            examples["text"],
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        
        # Replace padding token id with -100 for loss calculation
        labels["input_ids"] = [
            [(l if l != self.tokenizer.pad_token_id else -100) for l in label]
            for label in labels["input_ids"]
        ]
        
        return {
            "input_features": inputs.input_features,
            "labels": labels["input_ids"]
        }

# Initialize preprocessor
preprocessor = WhisperPreprocessor(data_loader.processor, config)

# Preprocess datasets
print("🔄 Preprocessing datasets...")
processed_datasets = datasets.map(
    preprocessor.preprocess_function,
    batched=True,
    batch_size=8,
    remove_columns=datasets["train"].column_names
)

print("✅ Preprocessing complete!")

In [ ]:
# Model initialization and training setup
class WhisperTrainer:
    """Whisper model trainer."""
    
    def __init__(self, config: WhisperConfig, processor):
        self.config = config
        self.processor = processor
        self.model = None
        self.trainer = None
        
    def initialize_model(self):
        """Initialize Whisper model for fine-tuning."""
        print(f"🚀 Loading {self.config.model_name}...")
        
        self.model = WhisperForConditionalGeneration.from_pretrained(
            self.config.model_name,
            cache_dir=self.config.cache_dir
        )
        
        # Set language and task
        self.model.generation_config.language = self.config.language
        self.model.generation_config.task = self.config.task
        self.model.generation_config.forced_decoder_ids = None
        
        # Enable gradient checkpointing for memory efficiency
        if self.config.gradient_checkpointing:
            self.model.gradient_checkpointing_enable()
        
        print(f"✅ Model loaded: {self.model.num_parameters():,} parameters")
        
    def setup_training(self, train_dataset, eval_dataset):
        """Setup training arguments and trainer."""
        
        training_args = Seq2SeqTrainingArguments(
            output_dir=str(self.config.output_dir),
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            learning_rate=self.config.learning_rate,
            warmup_steps=self.config.warmup_steps,
            max_steps=self.config.max_steps,
            gradient_checkpointing=self.config.gradient_checkpointing,
            fp16=self.config.fp16,
            eval_strategy="steps",  # FIXED: Changed from evaluation_strategy
            save_strategy="steps",
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            logging_steps=self.config.logging_steps,
            report_to=["tensorboard"],
            load_best_model_at_end=True,
            metric_for_best_model="wer",
            greater_is_better=False,
            push_to_hub=False,
            predict_with_generate=True,
            generation_max_length=225,
            save_total_limit=2,
            remove_unused_columns=False,
        )
        
        # Data collator
        from transformers import DataCollatorSpeechSeq2SeqWithPadding
        data_collator = DataCollatorSpeechSeq2SeqWithPadding(
            processor=self.processor,
            decoder_start_token_id=self.model.config.decoder_start_token_id,
        )
        
        # Compute metrics
        wer_metric = evaluate.load("wer")
        
        def compute_metrics(pred):
            pred_ids = pred.predictions
            label_ids = pred.label_ids
            
            # Replace -100 with pad token id
            label_ids[label_ids == -100] = self.processor.tokenizer.pad_token_id
            
            # Decode predictions and references
            pred_str = self.processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
            label_str = self.processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
            
            # Compute WER
            wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
            
            return {"wer": wer}
        
        # Initialize trainer
        self.trainer = Seq2SeqTrainer(
            args=training_args,
            model=self.model,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            tokenizer=self.processor.feature_extractor,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )
        
        print("✅ Trainer initialized!")
    
    def train(self):
        """Start training."""
        print("🏋️ Starting training...")
        print(f"📊 Training on {torch.cuda.device_count()} GPU(s)")
        
        # Train model
        train_result = self.trainer.train()
        
        # Save final model
        self.trainer.save_model()
        self.processor.save_pretrained(self.config.output_dir)
        
        # Save training metrics
        with open(self.config.output_dir / "training_results.json", "w") as f:
            json.dump(train_result.metrics, f, indent=2)
        
        print("✅ Training complete!")
        return train_result

# Initialize and train
trainer = WhisperTrainer(config, data_loader.processor)
trainer.initialize_model()

# Check if we have data
if len(processed_datasets["train"]) > 0:
    trainer.setup_training(
        processed_datasets["train"],
        processed_datasets["validation"]
    )
    print(f"🎯 Ready to train with {len(processed_datasets['train'])} samples!")
    print("Uncomment the next line to start training:")
    print("# train_result = trainer.train()")
else:
    print("⚠️ No training data available")

In [ ]:
# Convert to Faster-Whisper format
class FasterWhisperConverter:
    """Convert trained model to Faster-Whisper format."""
    
    @staticmethod
    def convert_to_ct2(model_path: Path, output_path: Path, quantization: str = "int8"):
        """Convert Whisper model to CTranslate2 format."""
        print("🔄 Converting to Faster-Whisper format...")
        
        # Use ct2-transformers-converter
        import subprocess
        
        cmd = [
            "ct2-transformers-converter",
            "--model", str(model_path),
            "--output_dir", str(output_path),
            "--quantization", quantization,
            "--force"
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"✅ Model converted to {output_path}")
        else:
            print(f"❌ Conversion failed: {result.stderr}")
            
    @staticmethod
    def load_faster_whisper(model_path: Path, device: str = "cuda"):
        """Load Faster-Whisper model."""
        model = WhisperModel(
            str(model_path),
            device=device,
            compute_type="int8_float16" if device == "cuda" else "int8"
        )
        return model

# Convert trained model (after training)
# converter = FasterWhisperConverter()
# converter.convert_to_ct2(
#     config.output_dir,
#     config.output_dir / "faster_whisper",
#     quantization="int8"
# )

In [ ]:
# Evaluation and benchmarking
class WhisperEvaluator:
    """Evaluate Whisper model performance."""
    
    def __init__(self, model_path: Path, test_dataset, processor):
        self.model_path = model_path
        self.test_dataset = test_dataset
        self.processor = processor
        self.results = {}
        
    def evaluate_transformers_model(self):
        """Evaluate standard Transformers model."""
        print("📊 Evaluating Transformers Whisper...")
        
        model = WhisperForConditionalGeneration.from_pretrained(self.model_path)
        model.eval()
        
        if torch.cuda.is_available():
            model = model.cuda()
        
        predictions = []
        references = []
        
        with torch.no_grad():
            for item in tqdm(self.test_dataset):
                # Prepare input
                inputs = self.processor(
                    item["audio"],
                    sampling_rate=16000,
                    return_tensors="pt"
                ).input_features
                
                if torch.cuda.is_available():
                    inputs = inputs.cuda()
                
                # Generate
                predicted_ids = model.generate(inputs)
                transcription = self.processor.batch_decode(
                    predicted_ids,
                    skip_special_tokens=True
                )[0]
                
                predictions.append(transcription)
                references.append(item["text"])
        
        # Calculate metrics
        wer_metric = evaluate.load("wer")
        wer = wer_metric.compute(predictions=predictions, references=references)
        
        self.results["transformers"] = {
            "wer": wer,
            "predictions": predictions[:5]  # Save sample predictions
        }
        
        print(f"✅ WER: {wer:.2%}")
        
    def evaluate_faster_whisper(self):
        """Evaluate Faster-Whisper model."""
        print("⚡ Evaluating Faster-Whisper...")
        
        fw_path = self.model_path / "faster_whisper"
        if not fw_path.exists():
            print("❌ Faster-Whisper model not found. Run conversion first.")
            return
        
        model = WhisperModel(
            str(fw_path),
            device="cuda" if torch.cuda.is_available() else "cpu",
            compute_type="int8_float16" if torch.cuda.is_available() else "int8"
        )
        
        predictions = []
        references = []
        times = []
        
        for item in tqdm(self.test_dataset):
            # Load audio
            audio, _ = librosa.load(item["audio"], sr=16000)
            
            # Transcribe
            import time
            start = time.time()
            segments, info = model.transcribe(
                audio,
                beam_size=5,
                language="tw"
            )
            elapsed = time.time() - start
            
            transcription = " ".join([s.text for s in segments])
            
            predictions.append(transcription)
            references.append(item["text"])
            times.append(elapsed)
        
        # Calculate metrics
        wer_metric = evaluate.load("wer")
        wer = wer_metric.compute(predictions=predictions, references=references)
        
        self.results["faster_whisper"] = {
            "wer": wer,
            "avg_time": np.mean(times),
            "predictions": predictions[:5]
        }
        
        print(f"✅ WER: {wer:.2%}")
        print(f"⏱️ Avg inference time: {np.mean(times):.3f}s")
    
    def benchmark_speed(self, audio_path: str, num_runs: int = 10):
        """Benchmark inference speed."""
        print(f"\n🏃 Speed Benchmark ({num_runs} runs)...")
        
        # Load test audio
        audio, _ = librosa.load(audio_path, sr=16000)
        
        results = {}
        
        # Benchmark Transformers
        model_hf = WhisperForConditionalGeneration.from_pretrained(self.model_path)
        if torch.cuda.is_available():
            model_hf = model_hf.cuda()
        model_hf.eval()
        
        times_hf = []
        for _ in range(num_runs):
            inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt").input_features
            if torch.cuda.is_available():
                inputs = inputs.cuda()
            
            import time
            start = time.time()
            with torch.no_grad():
                _ = model_hf.generate(inputs)
            times_hf.append(time.time() - start)
        
        results["transformers"] = {
            "mean": np.mean(times_hf),
            "std": np.std(times_hf),
            "min": np.min(times_hf),
            "max": np.max(times_hf)
        }
        
        # Benchmark Faster-Whisper
        fw_path = self.model_path / "faster_whisper"
        if fw_path.exists():
            model_fw = WhisperModel(
                str(fw_path),
                device="cuda" if torch.cuda.is_available() else "cpu"
            )
            
            times_fw = []
            for _ in range(num_runs):
                start = time.time()
                _ = model_fw.transcribe(audio, beam_size=5)
                times_fw.append(time.time() - start)
            
            results["faster_whisper"] = {
                "mean": np.mean(times_fw),
                "std": np.std(times_fw),
                "min": np.min(times_fw),
                "max": np.max(times_fw)
            }
            
            # Calculate speedup
            speedup = results["transformers"]["mean"] / results["faster_whisper"]["mean"]
            results["speedup"] = speedup
        
        self.results["speed_benchmark"] = results
        
        print("\n📊 Speed Benchmark Results:")
        print(f"  Transformers: {results['transformers']['mean']:.3f}s ± {results['transformers']['std']:.3f}s")
        if "faster_whisper" in results:
            print(f"  Faster-Whisper: {results['faster_whisper']['mean']:.3f}s ± {results['faster_whisper']['std']:.3f}s")
            print(f"  ⚡ Speedup: {results['speedup']:.2f}x")
    
    def save_results(self):
        """Save evaluation results."""
        results_path = self.model_path / "evaluation_results.json"
        with open(results_path, "w") as f:
            json.dump(self.results, f, indent=2)
        print(f"\n💾 Results saved to {results_path}")

# Run evaluation (after training)
# evaluator = WhisperEvaluator(
#     config.output_dir,
#     processed_datasets["test"],
#     data_loader.processor
# )
# evaluator.evaluate_transformers_model()
# evaluator.evaluate_faster_whisper()
# evaluator.save_results()

In [ ]:
# Interactive testing
class InteractiveTester:
    """Interactive testing interface."""
    
    def __init__(self, model_path: Path):
        self.model_path = model_path
        self.model = None
        self.processor = WhisperProcessor.from_pretrained(model_path)
        
    def load_model(self, use_faster_whisper: bool = True):
        """Load model for testing."""
        if use_faster_whisper:
            fw_path = self.model_path / "faster_whisper"
            if fw_path.exists():
                self.model = WhisperModel(
                    str(fw_path),
                    device="cuda" if torch.cuda.is_available() else "cpu"
                )
                print("✅ Faster-Whisper model loaded")
            else:
                print("❌ Faster-Whisper model not found")
        else:
            self.model = WhisperForConditionalGeneration.from_pretrained(self.model_path)
            if torch.cuda.is_available():
                self.model = self.model.cuda()
            self.model.eval()
            print("✅ Transformers model loaded")
    
    def transcribe_file(self, audio_path: str) -> str:
        """Transcribe audio file."""
        # Load audio
        audio, _ = librosa.load(audio_path, sr=16000)
        
        if isinstance(self.model, WhisperModel):
            # Faster-Whisper
            segments, _ = self.model.transcribe(audio, beam_size=5, language="tw")
            return " ".join([s.text for s in segments])
        else:
            # Transformers
            inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt").input_features
            if torch.cuda.is_available():
                inputs = inputs.cuda()
            
            with torch.no_grad():
                predicted_ids = self.model.generate(inputs)
            
            return self.processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    
    def test_samples(self):
        """Test on sample sentences."""
        print("\n🎤 Testing on sample audio...")
        
        # Test samples (would need actual audio files)
        test_files = [
            "../data/test/sample1.wav",
            "../data/test/sample2.wav",
        ]
        
        for audio_file in test_files:
            if Path(audio_file).exists():
                print(f"\n📁 File: {audio_file}")
                transcription = self.transcribe_file(audio_file)
                print(f"📝 Transcription: {transcription}")

# Interactive testing
# tester = InteractiveTester(config.output_dir)
# tester.load_model(use_faster_whisper=True)
# tester.test_samples()

## Summary

This notebook provides a complete pipeline for:

1. **Data Loading**: Loads Twi datasets from multiple sources
2. **Preprocessing**: Prepares audio and text for Whisper training
3. **Training**: Fine-tunes Whisper models on Twi data
4. **Optimization**: Converts to Faster-Whisper format for speed
5. **Evaluation**: Comprehensive evaluation with WER metrics
6. **Benchmarking**: Speed comparisons between implementations

### Key Features:
- Support for multiple Whisper model sizes
- Automatic mixed precision training
- CTranslate2 optimization for production
- Comprehensive evaluation metrics
- Interactive testing interface

### Next Steps:
1. Prepare more Twi training data
2. Run full training on GPU
3. Convert to Faster-Whisper format
4. Deploy for production use